In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t073'
task_json_path = f'{task_directory}/task.json'

with open(task_json_path, 'r') as file:
    task_data = json.load(file)

task_data

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert) and was triggered on April 20, 2026, at 11:21:54 (UTC+8). The alert time window spans from 11:20:01 to 11:23:54 (UTC+8). The affected entity is an APM operation named `inventory::/api/v1/inventory/{productId}`. The user has requested an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

{'task_id': 't073', 'task_version': '1.0', 'alert_event_id': '195ff92b621c1d2937d5fcd36e7747fa', 'alert_title': 'inventory接口流量下跌告警', 'alert_trigger_time': '2026-04-20T11:21:54+08:00', 'alert_window': {'start': '2026-04-20T11:20:01.627964+08:00', 'end': '2026-04-20T11:23:54+08:00'}, 'alert_entity': {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, 'prompt_text': '<code vibeops_object type="alert_event">inventory接口流量下跌告警<alert_event event_id="195ff92b621c1d2937d5fcd36e7747fa" trans_id="uf5oafbn2t329cj1d6ail2fvq4" rule_id="b7853e06-54b7-4409-9eea-9bb82fbfb252" rule_name="inventory接口流量下跌告警" entity_id="162799adab6799326676266a2c96cf3e" entity_name="inventory::/api/v1/inventory/{productId}" entity_type="apm.operation" entity_domain="apm" alert_time="2026-04-20T11:21:54+08:00" /></code> 帮我分析下根因。', 'workspace': 'rca-benchmark', 'region_id': 'cn-hongkong', 'available_modalities': ['metrics', 'logs', 'traces', 'events', 'alerts', 'topology'], 'scoring_note': 'Output contract (prediction_schema.json) and fault taxonomy (taxonomy.json) will be published in a follow-up release.'}```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_path)

# List all available KPIs
available_kpis = metrics_data['metric'].unique()
available_kpis

```
Out[2]:
```
The available KPIs include traffic-related metrics such as `request_count`, `error_count`, `error_rate`, and `avg_request_latency_seconds`. These metrics are relevant for analyzing traffic patterns and identifying potential issues.

The original code execution output of IPython Kernel is also provided below for reference:

['node_disk_usage_rate' 'node_pod_running_count' 'node_memory_usage_rate'
 'node_cpu_usage_rate' 'node_not_ready_status' 'node_ready_status'
 'deployment_desired_replicas' 'deployment_available_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_cpu_usage_total' 'deployment_memory_usage_total'
 'deployment_cpu_usage_vs_requests' 'deployment_cpu_usage_vs_limits'
 'cluster_node_count' 'cluster_number_of_pod' 'cluster_failed_node_count'
 'cluster_number_of_not_ready_pods' 'apiserver_non_2xx_read_request_qps'
 'apiserver_non_2xx_write_request_qps' 'apiserver_concurrent_connections'
 'apiserver_requests_per_second' 'namespace_number_of_not_ready_pods'
 'namespace_number_of_pods' 'slow_count' 'error_count' 'request_count'
 'avg_request_latency_seconds' 'arms_jvm_gc_total'
 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'error' 'slow' 'error_rate']```
```

In [ ]:
# In[3]:

import numpy as np
from pytz import timezone
from datetime import datetime

# Convert 'time' and 'value' columns to numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Filter metrics for the required KPIs
kpi_metrics = metrics_data[
    metrics_data['metric'].isin(['request_count', 'error_count', 'error_rate', 'avg_request_latency_seconds'])
]

# Calculate global P95 thresholds for each metric
p95_thresholds = kpi_metrics.groupby('metric')['value'].quantile(0.95)

# Convert alert window to timestamps in microseconds (UTC+8)
alert_start = int(datetime.fromisoformat('2026-04-20T11:20:01.627964+08:00').timestamp() * 1_000_000)
alert_end = int(datetime.fromisoformat('2026-04-20T11:23:54+08:00').timestamp() * 1_000_000)

# Filter data within the alert time window
filtered_data = kpi_metrics[(kpi_metrics['time'] >= alert_start) & (kpi_metrics['time'] <= alert_end)]

p95_thresholds, filtered_data

```
Out[3]:
```
The global P95 thresholds for the selected metrics across all components are as follows:
- `avg_request_latency_seconds`: 0.207 seconds
- `error_count`: 302
- `error_rate`: 0.143
- `request_count`: 11,686.7

Within the alert time window (11:20:01 to 11:23:54 UTC+8), 642 rows of relevant data were filtered for further analysis. This data includes metrics such as `request_count`, `error_count`, `error_rate`, and `avg_request_latency_seconds` for various components.

The original code execution output of IPython Kernel is also provided below for reference:

(metric
avg_request_latency_seconds        0.207444
error_count                      302.000000
error_rate                         0.142627
request_count                  11686.700000
Name: value, dtype: float64,                    time domain     entity_set                         entity_id                      entity_name       metric     value                            metric_set_id service
54127  1776655231000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  error_count  2.000000  apm@metric_set@apm.metric.apm.operation    None
54128  1776655261000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  error_count  4.000000  apm@metric_set@apm.metric.apm.operation    None
54129  1776655291000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  error_count  5.000000  apm@metric_set@apm.metric.apm.operation    None
54130  1776655321000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  error_count  8.000000  apm@metric_set@apm.metric.apm.operation    None
54131  1776655351000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  error_count  8.000000  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...            ...                               ...                              ...          ...       ...                                      ...     ...
82013  1776655291000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6   error_rate  0.001528   apm@metric_set@apm.metric.apm.instance    None
82014  1776655321000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6   error_rate  0.002496   apm@metric_set@apm.metric.apm.instance    None
82015  1776655351000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6   error_rate  0.002091   apm@metric_set@apm.metric.apm.instance    None
82016  1776655381000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6   error_rate  0.001663   apm@metric_set@apm.metric.apm.instance    None
82017  1776655411000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6   error_rate  0.002177   apm@metric_set@apm.metric.apm.instance    None

[642 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Identify anomalies by comparing each metric against its global P95 threshold
anomalies = filtered_data[
    (filtered_data['metric'] == 'request_count') & (filtered_data['value'] > p95_thresholds['request_count']) |
    (filtered_data['metric'] == 'error_count') & (filtered_data['value'] > p95_thresholds['error_count']) |
    (filtered_data['metric'] == 'error_rate') & (filtered_data['value'] > p95_thresholds['error_rate']) |
    (filtered_data['metric'] == 'avg_request_latency_seconds') & (filtered_data['value'] > p95_thresholds['avg_request_latency_seconds'])
]

# List components and time points where anomalies occurred
anomalies_details = anomalies[['time', 'entity_name', 'metric', 'value']]
anomalies_details

```
Out[4]:
```
Anomalies were detected in the `avg_request_latency_seconds` metric, where the values exceeded the global P95 threshold (0.207 seconds). These anomalies occurred across multiple components, including `frontend-proxy::ingress`, `frontend::POST`, `checkout::/oteldemo.CheckoutService/PlaceOrder`, and various instances of `frontend` and `frontend-proxy`. The anomalies were observed at multiple time points within the alert window, indicating potential performance issues in these components.

The original code execution output of IPython Kernel is also provided below for reference:

time                                     entity_name                       metric     value
58787  1776655291000000                         frontend-proxy::ingress  avg_request_latency_seconds  0.343438
58788  1776655321000000                         frontend-proxy::ingress  avg_request_latency_seconds  0.487747
58789  1776655351000000                         frontend-proxy::ingress  avg_request_latency_seconds  0.455328
58790  1776655381000000                         frontend-proxy::ingress  avg_request_latency_seconds  0.419549
58791  1776655411000000                         frontend-proxy::ingress  avg_request_latency_seconds  0.425710
59786  1776655231000000                                  frontend::POST  avg_request_latency_seconds  0.229153
59787  1776655261000000                                  frontend::POST  avg_request_latency_seconds  2.343647
59788  1776655291000000                                  frontend::POST  avg_request_latency_seconds  2.499602
59789  1776655321000000                                  frontend::POST  avg_request_latency_seconds  2.644817
59790  1776655351000000                                  frontend::POST  avg_request_latency_seconds  2.654729
59791  1776655381000000                                  frontend::POST  avg_request_latency_seconds  2.625296
59792  1776655411000000                                  frontend::POST  avg_request_latency_seconds  2.609437
60287  1776655231000000  checkout::/oteldemo.CheckoutService/PlaceOrder  avg_request_latency_seconds  0.428155
61572  1776655231000000                    frontend::POST /api/checkout  avg_request_latency_seconds  0.418935
66806  1776655291000000                 frontend-proxy-67d64dc4f8-8c99b  avg_request_latency_seconds  0.314966
66807  1776655321000000                 frontend-proxy-67d64dc4f8-8c99b  avg_request_latency_seconds  0.488479
66808  1776655351000000                 frontend-proxy-67d64dc4f8-8c99b  avg_request_latency_seconds  0.467188
66809  1776655381000000                 frontend-proxy-67d64dc4f8-8c99b  avg_request_latency_seconds  0.367867
66810  1776655411000000                 frontend-proxy-67d64dc4f8-8c99b  avg_request_latency_seconds  0.342296
67284  1776655231000000                          checkout-5489989-q7ws7  avg_request_latency_seconds  0.428155
67395  1776655291000000                 frontend-proxy-67d64dc4f8-xz2fd  avg_request_latency_seconds  0.362061
67396  1776655321000000                 frontend-proxy-67d64dc4f8-xz2fd  avg_request_latency_seconds  0.629495
67397  1776655351000000                 frontend-proxy-67d64dc4f8-xz2fd  avg_request_latency_seconds  0.653436
67398  1776655381000000                 frontend-proxy-67d64dc4f8-xz2fd  avg_request_latency_seconds  0.678680
67399  1776655411000000                 frontend-proxy-67d64dc4f8-xz2fd  avg_request_latency_seconds  0.704719
69511  1776655261000000                 frontend-proxy-67d64dc4f8-4ptg6  avg_request_latency_seconds  0.210919
69512  1776655291000000                 frontend-proxy-67d64dc4f8-4ptg6  avg_request_latency_seconds  0.362814
69513  1776655321000000                 frontend-proxy-67d64dc4f8-4ptg6  avg_request_latency_seconds  0.403516
69514  1776655351000000                 frontend-proxy-67d64dc4f8-4ptg6  avg_request_latency_seconds  0.355021
69515  1776655381000000                 frontend-proxy-67d64dc4f8-4ptg6  avg_request_latency_seconds  0.346195
69516  1776655411000000                 frontend-proxy-67d64dc4f8-4ptg6  avg_request_latency_seconds  0.376817
69763  1776655261000000                        frontend-86465b665-94wcz  avg_request_latency_seconds  0.768170
69764  1776655291000000                        frontend-86465b665-94wcz  avg_request_latency_seconds  0.857407
69765  1776655321000000                        frontend-86465b665-94wcz  avg_request_latency_seconds  0.996639
69766  1776655351000000                        frontend-86465b665-94wcz  avg_request_latency_seconds  1.011716
69767  1776655381000000                        frontend-86465b665-94wcz  avg_request_latency_seconds  0.977835
69768  1776655411000000                        frontend-86465b665-94wcz  avg_request_latency_seconds  0.944665```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_path)

# Convert 'startTime', 'endTime', and 'duration' columns to numeric for calculations
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces within the alert time window (convert to nanoseconds for comparison)
alert_start_ns = alert_start * 1_000
alert_end_ns = alert_end * 1_000
filtered_traces = traces_data[
    (traces_data['startTime'] >= alert_start_ns) & (traces_data['endTime'] <= alert_end_ns)
]

# Identify traces related to the affected operation
affected_operation_traces = filtered_traces[
    filtered_traces['spanName'].str.contains('inventory::/api/v1/inventory/{productId}', case=False, na=False)
]

# Analyze the call chain to determine the most downstream faulty service
downstream_faulty_service = affected_operation_traces[['traceId', 'spanName', 'serviceName', 'statusCode', 'duration']]
downstream_faulty_service_sorted = downstream_faulty_service.sort_values(by=['traceId', 'duration'], ascending=False)

downstream_faulty_service_sorted

```
Out[5]:
```
No traces related to the affected operation `inventory::/api/v1/inventory/{productId}` were found within the alert time window (11:20:01 to 11:23:54 UTC+8). This indicates that there were no recorded spans for this operation during the specified period, making it impossible to analyze the call chain or identify the most downstream faulty service based on trace data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanName, serviceName, statusCode, duration]
Index: []```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_path)

# Convert '_time_' column to numeric if necessary and filter logs within the alert time window
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) & (logs_data['_time_'] <= alert_end)
]

# Search for logs containing error messages or references to the affected operation or related components
error_logs = filtered_logs[
    filtered_logs['content'].str.contains(
        'error|inventory::/api/v1/inventory/{productId}|frontend|frontend-proxy|checkout',
        case=False, na=False
    )
]

error_logs

```
Out[6]:
```
No logs containing error messages or references to the affected operation `inventory::/api/v1/inventory/{productId}` or related components such as `frontend`, `frontend-proxy`, and `checkout` were found within the alert time window (11:20:01 to 11:23:54 UTC+8). This indicates that there were no relevant log entries recorded during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```